<h1 style="text-align: center">Notebook 1: Introducción a MONAI</h1>
<h2 style="text-align: center">Parte 2</h2>

<p style="text-align: center">Unidad 1: Fundamentos de IA</p>
<p style="text-align: center">Este cuaderno nos introduce en MONAI Core. Veremos algunos ejemplos prácticos sobre transformaciones, cargadores de datos, caché y redes.</p>

### Comprobar el acceso a la GPU

Si ejecutan **!nvidia-smi** en una celda, podrán comprobar el tipo de hardware al que tiene acceso.

In [ ]:
!nvidia-smi

### Paquetes necesarios para ejecutar en Colab

Ejecutar la siguiente celda para instalar MONAI la primera vez que ejecute este notebook en Colab:

In [ ]:
!python -c "import monai" || pip install -qU "monai[ignite, nibabel, torchvision, tqdm]==1.2.0"
!python -c "import sklearn" || pip install -qU "scikit-learn"

## Entrenamiento de extremo a extremo con PyTorch

Hemos visto mucho material y ahora es momento de aplicar lo aprendido en un ejemplo de extremo a extremo usando el paradigma básico de PyTorch. Cubriremos:

1. **Configurar nuestro dataset y explorar los datos**
2. **Preparar datasets y transformaciones**
3. **Definir la red y crear nuestro bucle de entrenamiento en MONAI/PyTorch**
4. **Evaluar el modelo y entender los resultados**

### Importaciones

Empecemos importando nuestras dependencias.

In [ ]:
import os
import shutil
import tempfile
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import PIL

import torch
from torch.utils.data import DataLoader
import monai

from monai.apps import download_and_extract
from monai.config import print_config
from monai.metrics import ROCAUCMetric
from monai.data import decollate_batch, partition_dataset_classes
from monai.networks.nets import DenseNet121
from monai.transforms import (
    EnsureChannelFirst, Compose,LoadImage,RandFlip, RandRotate,
    RandZoom, ScaleIntensity, Activations, AsDiscrete, EnsureType
)
from monai.utils import set_determinism

### 1. Configurar nuestro dataset y explorar los datos

#### Configurar el directorio de datos

Crearemos un directorio temporal para todos los datos de MONAI que vamos a utilizar, llamado `MONAI_DATA_DIRECTORY`.

In [ ]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

#### Descargar el dataset MedNIST
El dataset `MedNIST` se recopiló a partir de varios conjuntos de [TCIA](https://wiki.cancerimagingarchive.net/display/Public/Data+Usage+Policies+and+Restrictions),
[el RSNA Bone Age Challenge](http://rsnachallenges.cloudapp.net/competitions/4),
y [el dataset NIH Chest X-ray](https://cloud.google.com/healthcare/docs/resources/public-datasets/nih-chest).

El dataset es cortesía del [Dr. Bradley J. Erickson M.D., Ph.D.](https://www.mayo.edu/research/labs/radiology-informatics/overview) (Departamento de Radiología, Mayo Clinic)
bajo la licencia Creative Commons [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Si usa MedNIST, por favor cite la fuente.

Descargar este dataset y extraerlo en el directorio temporal:

In [ ]:
resource = "https://www.dropbox.com/s/5wwskxctvcxiuea/MedNIST.tar.gz?dl=1"
md5 = "0bc7306e7427e00ad1c5526a6677552d"

compressed_file = os.path.join(root_dir, "MedNIST.tar.gz")
data_dir = os.path.join(root_dir, "MedNIST")
if not os.path.exists(data_dir):
    download_and_extract(resource, compressed_file, root_dir, md5)

#### Establecer entrenamiento determinista para reproducibilidad

`set_determinism` ([ver documentación](https://monai.readthedocs.io/en/stable/utils.html#monai.utils.misc.set_determinism)) establecerá las semillas aleatorias en NumPy y PyTorch para garantizar reproducibilidad. Más adelante veremos que es necesario hacer algunas cosas adicionales para asegurar reproducibilidad en un notebook Jupyter. Por ahora, también instanciamos un valor de semilla que usaremos después.

In [ ]:
set_determinism(seed=0)
rseed = 12345678

#### Leer los nombres de archivo de imagen desde las carpetas del dataset

Al usar un dataset conviene entender lo básico sobre las imágenes, las etiquetas y más. Comenzaremos mostrando algunas estadísticas básicas de MedNIST.

Luego definiremos nuestra propia clase `MedNISTDataset` con fines ilustrativos, aunque MONAI proporciona una versión más robusta.

MedNIST tiene 6 carpetas diferentes que representan 6 categorías: Hand, AbdomenCT, CXR, ChestCT, BreastMRI, HeadCT. Usaremos cada una de estas categorías como nombres de etiqueta.

In [ ]:
subdirs = sorted(filter(os.path.isdir, glob(f"{data_dir}/*")))
class_names = list(map(os.path.basename, subdirs))
num_class = len(class_names)

image_files = [sorted(glob(f"{d}/*")) for d in subdirs]

num_each = [len(image_files[i]) for i in range(num_class)]
image_files_list = sum(image_files, [])
image_class = sum([[i] * n for i, n in enumerate(num_each)], [])

num_total = len(image_class)
image_width, image_height = PIL.Image.open(image_files_list[0]).size

print(f"Total image count: {num_total}")
print(f"Image dimensions: {image_width} x {image_height}")
print(f"Label names: {class_names}")
print(f"Label counts: {num_each}")

#### Seleccionar imágenes aleatorias del dataset para visualizar y comprobar

Queremos entender cómo son las imágenes que usamos, así que comenzaremos visualizando algunas imágenes aleatorias.

In [ ]:
plt.subplots(2, 5, figsize=(8, 4))
for i, k in enumerate(np.random.randint(num_total, size=10)):
    im = PIL.Image.open(image_files_list[k])
    plt.subplot(2, 5, i + 1)
    plt.axis("off")
    plt.title(class_names[image_class[k]])
    plt.imshow(np.array(im), cmap="gray", vmin=0, vmax=255)

## 2. Preparar datasets y transformaciones

Queremos dividir los datos en 3 conjuntos diferentes: entrenamiento, validación y prueba. Usaremos una proporción 80/10/10.

[Documentación de MONAI](https://monai.readthedocs.io/en/stable/api.html)


<div class="alert alert-block alert-info">
    <b>Actividad</b>

- Utilizar la función `partition_dataset_classes` para dividir adecuadamente el dataset en partes
- Asegurarse de usar la variable `rseed` definida anteriormente en `partition_dataset_classes` para garantizar reproducibilidad
- Usar una división 80/10/10 para el dataset
- Usar las partes para crear las nuevas listas de imágenes y de etiquetas
- Crear subconjuntos `train`, `val` y `test`, cada uno con una lista de imágenes (p. ej. `train_x`) y una lista de etiquetas (`train_y`)

</div>

In [ ]:
parts = partition_dataset_classes(???)
    
# Completar para obtener los conjuntos de entrenamiento, validación y prueba
???

print(f"Training count: {len(train_x)}, Validation count: {len(val_x)}, Test count: {len(test_x)}")

#### Definir transformaciones MONAI, Dataset y Dataloader para preprocesar datos

Definiremos nuestra secuencia de transformaciones con `Compose`, en la que cargaremos la imagen, añadiremos un canal, escalaremos su intensidad y utilizaremos algunas funciones aleatorias.

[Documentación de MONAI](https://docs.monai.io/en/stable/)

<div class="alert alert-block alert-info">
    <b>Actividad</b>
    
A continuación definirá el `Compose` de transformaciones para los datos de entrenamiento. Necesitará las siguientes transformaciones:
    
- Cargar imagen (Load Image)
- Asegurar primer canal (Ensure Channel First)
- Escalar intensidad (Scale Intensity)
- Rotación, volteo y zoom aleatorios

</div>

:::{.callout-note title="Nota"}
Pytorch define los tensores de la siguiente manera:
- **2D**: [B, C, H, W] donde B es el tamaño del batch, C es el número de canales, H es la altura y W es el ancho. Esto significa, por ejemplo, que cada imagen debe tener un canal (C=1) y que las imágenes deben ser de tamaño 28x28 (H=28, W=28). MONAI proporciona transformaciones para asegurarse de que las imágenes tengan el formato correcto.

- **3D**: [B, C, D, H, W] donde B es el tamaño del batch, C es el número de canales, D es la profundidad, H es la altura y W es el ancho. Esto significa, por ejeemplo, que cada imagen debe tener un canal (C=1) y que las imágenes deben ser de tamaño 64x64x64 (D=64, H=64, W=64). MONAI proporciona transformaciones para asegurarse de que las imágenes tengan el formato correcto.
:::

In [ ]:
rseed = 12345678

train_transforms = Compose(
    [ ??? ]
)

val_transforms = Compose(
    [ ??? ]
)

act = Compose([Activations(softmax=True)])
to_onehot = Compose([AsDiscrete(to_onehot=num_class)])

In [ ]:
### Transform Test Functionality
plt.subplots(1, 3, figsize=(4, 4))
for i in range(0,3):
    test_output = train_transforms(image_files_list[i])
    #test_output = val_transforms(image_files_list[i])    
    
    arr = np.array(test_output[0])
    plt.subplot(1, 3, i + 1)
    plt.xlabel(class_names[image_class[k]])
    plt.imshow(arr, cmap="gray", vmin=0, vmax=1)
plt.tight_layout()
plt.show()

#### Inicializar los datasets y loaders para los conjuntos de entrenamiento, validación y prueba
 * Defina un dataset simple, que llamaremos `MedNISTDataset`, que agrupe:
   * Imágenes
   * Etiquetas
   * Las transformaciones que se aplicarán a las imágenes y etiquetas
 * Cree tres instancias de este dataset:
   * Una para entrenamiento
   * Una para validación
   * Una para prueba
   
Usaremos un tamaño de lote de 512 y 10 workers para cargar los datos.

In [ ]:
as asdasdbatch_size = 512 #En caso de que se produzca un error de memoria, reducir el tamaño del batch. Por ejemplo, batch_size = 256 
 num_workers = 10

class MedNISTDataset(torch.utils.data.Dataset):
    def __init__(self, image_files, labels, transforms):
        self.image_files = image_files
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        return self.transforms(self.image_files[index]), self.labels[index]


train_ds = MedNISTDataset(train_x, train_y, train_transforms)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)

val_ds = MedNISTDataset(val_x, val_y, val_transforms)
val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers)

test_ds = MedNISTDataset(test_x, test_y, val_transforms)
test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=num_workers)

#### Definir la red y el optimizador

Nos aseguraremos de configurar los valores relevantes para obtener el dispositivo y instanciar la red, la función de pérdida y el optimizador.

[Documentación de MONAI](https://docs.monai.io/en/stable/)

<div class="alert alert-block alert-info">
    <b>Actividad</b>
    
Haga lo siguiente:
1. Obtenga el dispositivo GPU desde `torch`
2. Instancie `DenseNet121` ([ver documentación](https://monai.readthedocs.io/en/stable/networks.html#densenet121)). Deberá establecer la dimensión espacial a 2, canales de entrada a 1 y canales de salida al número de clases obtenido en la exploración de datos.
3. Instancie la función de pérdida `CrossEntropy` de Torch
4. Instancie el optimizador `Adam` de Torch. Probar diferentes tasas de aprendizaje, como 1e-5, 1e-4 y 1e-3, para ver cómo afectan el entrenamiento. 
5. Probar otros optimizadores como `SGD` y `RMSprop` para ver cómo afectan el entrenamiento.
</div>

In [ ]:
device = ???
net = ???
loss_function = ???
optimizer = ???

#### Entrenamiento de la red
Aquí implementamos manualmente un bucle de entrenamiento básico en PyTorch:
 * bucle de entrenamiento estándar en PyTorch
   * recorrer cada época de entrenamiento en lotes
   * después de cada época, ejecutar una pasada de validación para evaluar la red
   * si muestra mejor rendimiento, guardar los pesos del modelo
 * más adelante revisaremos bucles de entrenamiento en un estilo más integrado con Ignite / MONAI

In [ ]:
epoch_num = 4
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = list()
metric_values = list()
auc_metric = ROCAUCMetric()

for epoch in range(epoch_num):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{epoch_num}")

    epoch_loss = 0
    step = 1

    steps_per_epoch = len(train_ds) // train_loader.batch_size

    # put the network in train mode; this tells the network and its modules to
    # enable training elements such as normalisation and dropout, where applicable
    net.train()
    for batch_data in train_loader:

        # move the data to the GPU
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)

        # prepare the gradients for this step's back propagation
        optimizer.zero_grad()
        
        # run the network forwards
        outputs = net(inputs)
        
        # run the loss function on the outputs
        loss = loss_function(outputs, labels)
        
        # compute the gradients
        loss.backward()
        
        # tell the optimizer to update the weights according to the gradients
        # and its internal optimisation strategy
        optimizer.step()

        epoch_loss += loss.item()
        print(f"{step}/{len(train_ds) // train_loader.batch_size + 1}, training_loss: {loss.item():.4f}")
        step += 1

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # after each epoch, run our metrics to evaluate it, and, if they are an improvement,
    # save the model out
    
    # switch off training features of the network for this pass
    net.eval()

    # 'with torch.no_grad()' switches off gradient calculation for the scope of its context
    with torch.no_grad():
        # create lists to which we will concatenate the the validation results
        preds = list()
        labels = list()

        # iterate over each batch of images and run them through the network in evaluation mode
        for val_data in val_loader:
            val_images, val_labels = val_data[0].to(device), val_data[1].to(device)

            # run the network
            val_pred = net(val_images)

            preds.append(val_pred)
            labels.append(val_labels)

        # concatenate the predicted labels with each other and the actual labels with each other
        y_pred = torch.cat(preds)
        y = torch.cat(labels)

        # we are using the area under the receiver operating characteristic (ROC) curve to determine
        # whether this epoch has improved the best performance of the network so far, in which case
        # we save the network in this state
        y_onehot = [to_onehot(i) for i in decollate_batch(y, detach=False)]        
        y_pred_act = [act(i) for i in decollate_batch(y_pred)]
        
        auc_metric(y_pred_act, y_onehot)
        auc_value = auc_metric.aggregate()
        auc_metric.reset()
        metric_values.append(auc_value)
        
        acc_value = torch.eq(y_pred.argmax(dim=1), y)
        acc_metric = acc_value.sum().item() / len(acc_value)
        
        if auc_value > best_metric:
            best_metric = auc_value
            best_metric_epoch = epoch + 1
            torch.save(net.state_dict(), os.path.join(root_dir, "best_metric_model.pth"))
            print("saved new best metric network")
            
        print(
            f"current epoch: {epoch + 1} current AUC: {auc_value:.4f} /"
            f" current accuracy: {acc_metric:.4f} best AUC: {best_metric:.4f} /"
            f" at epoch: {best_metric_epoch}"
        )

print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")

#### Graficar la pérdida y la métrica

Al terminar el entrenamiento queremos visualizar la pérdida y la precisión.

In [ ]:
plt.figure("train", (12, 6))
plt.subplot(1, 2, 1)
plt.title("Epoch Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.subplot(1, 2, 2)
plt.title("Val AUC")
x = [(i + 1) for i in range(len(metric_values))]
y = metric_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.show()

## **4. Evaluar su modelo y entender los resultados**

Tras el entrenamiento y la validación, ahora tenemos el mejor modelo según el conjunto de validación. Ahora debemos evaluar el modelo en el conjunto de prueba para comprobar si el modelo final es robusto y no está sobreajustado. Usaremos estas predicciones para generar un informe de clasificación.

In [ ]:
net.load_state_dict(torch.load(os.path.join(root_dir, "best_metric_model.pth")))
net.eval()
y_true = list()
y_pred = list()

with torch.no_grad():
    for test_data in test_loader:
        test_images, test_labels = (
            test_data[0].to(device),
            test_data[1].to(device),
        )
        pred = net(test_images).argmax(dim=1)
        
        for i in range(len(pred)):
            y_true.append(test_labels[i].item())
            y_pred.append(pred[i].item())

#### Análisis básico - informe de clasificación

Utilizaremos el `classification_report` de scikit-learn para obtener precisión, recall y f1-score por categoría.

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

#### Análisis básico - matriz de confusión

También crearemos una matriz de confusión para entender mejor los casos de fallo.

In [ ]:
from sklearn.metrics import confusion_matrix

cmat = confusion_matrix(y_true, y_pred)

cax = plt.matshow(cmat, cmap="turbo", interpolation="nearest")
plt.colorbar(cax)

cax.axes.set_xticks(list(range(len(class_names))), class_names, rotation=270)
cax.axes.set_yticks(list(range(len(class_names))), class_names)

plt.show()

## **Resumen**

En este cuaderno hemos recorrido un flujo de trabajo de extremo a extremo para entrenar el dataset MedNIST usando una red densenet121. En el proceso, usted:
- Aprendimos sobre los datos de MedNIST 
- Visualizamos los datos para entender las imágenes
- Se configuró los datasets para el entrenamiento del modelo
- Se definieron transformaciones, datasets, la red y los optimizadores
- Se entrenó un modelo densenet y guardó el mejor modelo según la métrica de validación
- Se graficaron los resultados de entrenamiento
- Se evaluaron el modelo en el conjunto de prueba
- Se ejecutaron las predicciones finales y generó un informe de clasificación para entender los resultados
